In [ ]:
"""
KMZ Path Stitcher - Modified from Elevation Profile Extraction v7
Stitches together multiple path segments from KMZ files and smooths jagged corners.
NO ELEVATION EXTRACTION - Only path stitching and waypoint preservation.
"""

import os
import re
import math
import zipfile
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
import numpy as np

try:
    from scipy.interpolate import splprep, splev
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("Warning: scipy not available. Smoothing will be disabled.")

try:
    import simplekml
    SIMPLEKML_AVAILABLE = True
except ImportError:
    SIMPLEKML_AVAILABLE = False
    print("Error: simplekml not installed. Install with: pip install simplekml")

# ==================== CONFIGURATION ====================

# Input/Output
INPUT_KMZ = r"C:\Users\peckb\Documents\APEx\Cincinnati Metro\Line 6.kmz"
OUTPUT_DIR = Path(INPUT_KMZ).parent
OUTPUT_KMZ = OUTPUT_DIR / f"{Path(INPUT_KMZ).stem}_stitched.kmz"

# Smoothing settings
ENABLE_SMOOTHING = False
SMOOTHING_FACTOR = 5.0  # Lower = more aggressive smoothing (0 = interpolation)
SMOOTHING_POINTS_PER_KM = 50  # Points to generate per km of route
MIN_CORNER_ANGLE_DEG = 160  # Only smooth corners sharper than this angle

# ==================== DATA STRUCTURES ====================

@dataclass
class RoutePoint:
    """Represents a point along the route"""
    index: int
    lat: float
    lon: float
    distance_m: float
    elevation: Optional[float] = None  # Not used in this version
    data_source: str = "none"
    is_interpolated: bool = False

# ==================== UTILITY FUNCTIONS ====================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance between two points on Earth in meters"""
    R = 6371000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def calculate_angle(p1: Tuple[float, float], p2: Tuple[float, float], 
                   p3: Tuple[float, float]) -> float:
    """
    Calculate angle at point p2 formed by p1-p2-p3 in degrees.
    Returns angle in range [0, 180]
    """
    v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]])
    v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]])
    
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    angle_rad = np.arccos(cos_angle)
    return math.degrees(angle_rad)

# ==================== KMZ EXTRACTION (from original v7 code) ====================

def extract_coordinates_from_kmz_with_path_fixing(kmz_path: str) -> List[Tuple[float, float]]:
    """
    Extract coordinates from KMZ file with automatic path reversal detection and fixing.
    This function matches the original v7 code structure.
    """
    print(f"Extracting coordinates from {os.path.basename(kmz_path)}...")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kmz.extractall(temp_dir)
            
            kml_file = None
            for file in os.listdir(temp_dir):
                if file.endswith('.kml'):
                    kml_file = os.path.join(temp_dir, file)
                    break
            
            if not kml_file:
                raise ValueError("No KML file found in KMZ")
            
            tree = ET.parse(kml_file)
            root = tree.getroot()
    
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    if not root.findall('.//kml:coordinates', ns):
        ns = {}
    
    all_paths = []
    path_names = []
    
    # Find all Placemarks that contain LineStrings
    if ns:
        placemarks = root.findall('.//kml:Placemark', ns)
    else:
        placemarks = root.findall('.//Placemark')
    
    for placemark in placemarks:
        if ns:
            name_elem = placemark.find('kml:name', ns)
            linestring = placemark.find('.//kml:LineString', ns)
        else:
            name_elem = placemark.find('name')
            linestring = placemark.find('.//LineString')
        
        name = name_elem.text if name_elem is not None else "Unnamed"
        
        if linestring is not None:
            if ns:
                coords_elem = linestring.find('kml:coordinates', ns)
            else:
                coords_elem = linestring.find('coordinates')
            
            if coords_elem is not None:
                path_coordinates = []
                coords_text = coords_elem.text.strip()
                for coord in coords_text.split():
                    parts = coord.split(',')
                    if len(parts) >= 2:
                        lon = float(parts[0])
                        lat = float(parts[1])
                        path_coordinates.append((lat, lon))
                
                if path_coordinates:
                    all_paths.append(path_coordinates)
                    path_names.append(name)
                    print(f"  Found path '{name}' with {len(path_coordinates)} points")
    
    # If no placemarks found, fall back to looking for LineStrings directly
    if not all_paths:
        if ns:
            linestrings = root.findall('.//kml:LineString', ns)
        else:
            linestrings = root.findall('.//LineString')
        
        for i, linestring in enumerate(linestrings):
            if ns:
                coords_elem = linestring.find('.//kml:coordinates', ns)
            else:
                coords_elem = linestring.find('.//coordinates')
            
            if coords_elem is not None:
                path_coordinates = []
                coords_text = coords_elem.text.strip()
                for coord in coords_text.split():
                    parts = coord.split(',')
                    if len(parts) >= 2:
                        lon = float(parts[0])
                        lat = float(parts[1])
                        path_coordinates.append((lat, lon))
                
                if path_coordinates:
                    all_paths.append(path_coordinates)
                    path_names.append(f"Path {i+1}")
                    print(f"  Found path {i+1} with {len(path_coordinates)} points")
    
    if not all_paths:
        raise ValueError("No coordinate paths found in KMZ")
    
    print(f"\n  Total paths found: {len(all_paths)}")
    
    # Path reversal detection and fixing (from original v7 code)
    if len(all_paths) > 1:
        print("\n  Checking path continuity and detecting reversals...")
        
        working_paths = [path[:] for path in all_paths]
        
        for i in range(len(working_paths) - 1):
            current_path = working_paths[i]
            next_path = working_paths[i + 1]
            
            # Calculate all possible connection distances
            connections = {
                'normal': haversine_distance(
                    current_path[-1][0], current_path[-1][1],
                    next_path[0][0], next_path[0][1]
                ),
                'reverse_next': haversine_distance(
                    current_path[-1][0], current_path[-1][1],
                    next_path[-1][0], next_path[-1][1]
                ),
                'reverse_current': haversine_distance(
                    current_path[0][0], current_path[0][1],
                    next_path[0][0], next_path[0][1]
                ),
                'reverse_both': haversine_distance(
                    current_path[0][0], current_path[0][1],
                    next_path[-1][0], next_path[-1][1]
                )
            }
            
            best_connection = min(connections, key=connections.get)
            best_distance = connections[best_connection]
            
            if best_connection == 'reverse_next':
                working_paths[i + 1] = list(reversed(next_path))
                print(f"    Reversed path {i+2} '{path_names[i+1]}' (gap: {best_distance:.1f}m)")
            elif best_connection == 'reverse_current':
                working_paths[i] = list(reversed(current_path))
                print(f"    Reversed path {i+1} '{path_names[i]}' (gap: {best_distance:.1f}m)")
            elif best_connection == 'reverse_both':
                working_paths[i] = list(reversed(current_path))
                working_paths[i + 1] = list(reversed(next_path))
                print(f"    Reversed both paths {i+1} and {i+2} (gap: {best_distance:.1f}m)")
        
        # Flatten all paths into single coordinate list
        stitched_coordinates = []
        for path in working_paths:
            stitched_coordinates.extend(path)
    else:
        stitched_coordinates = all_paths[0]
    
    print(f"\n  Total stitched points: {len(stitched_coordinates)}")
    return stitched_coordinates

def extract_kmz_waypoints(kmz_path: str, exclude_elevation_waypoints: bool = True) -> List[Dict]:
    """
    Extract waypoint Placemarks from KMZ file, optionally excluding manual elevation waypoints.
    Matches original v7 code structure.
    """
    print(f"\nExtracting waypoints from {os.path.basename(kmz_path)}...")
    
    waypoints = []
    elevation_pattern = re.compile(r'ELEVATION\s*=\s*\d+(?:\.\d+)?\s*FT', re.IGNORECASE)
    
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kmz.extractall(temp_dir)
            
            kml_file = None
            for file in os.listdir(temp_dir):
                if file.endswith('.kml'):
                    kml_file = os.path.join(temp_dir, file)
                    break
            
            if not kml_file:
                raise ValueError("No KML file found in KMZ")
            
            tree = ET.parse(kml_file)
            root = tree.getroot()
    
    # Check for namespace
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}
    if not root.findall('.//kml:Point', ns):
        ns = {}
    
    # Extract ONLY Point placemarks (actual waypoints), NOT LineString vertices
    if ns:
        placemarks = root.findall('.//kml:Placemark', ns)
    else:
        placemarks = root.findall('.//Placemark')
    
    excluded_count = 0
    
    for placemark in placemarks:
        # Only process Placemarks with Points (waypoints), skip LineStrings
        if ns:
            point = placemark.find('.//kml:Point', ns)
            name_elem = placemark.find('kml:name', ns)
            desc_elem = placemark.find('kml:description', ns)
        else:
            point = placemark.find('.//Point')
            name_elem = placemark.find('name')
            desc_elem = placemark.find('description')
        
        if point is not None:
            waypoint_name = name_elem.text if name_elem is not None else "Waypoint"
            description = desc_elem.text if desc_elem is not None else ""
            
            # Check if this is a manual elevation waypoint and skip if requested
            if exclude_elevation_waypoints and elevation_pattern.search(waypoint_name):
                excluded_count += 1
                continue
            
            if ns:
                coords_elem = point.find('kml:coordinates', ns)
            else:
                coords_elem = point.find('coordinates')
            
            if coords_elem is not None:
                coords_text = coords_elem.text.strip()
                parts = coords_text.split(',')
                if len(parts) >= 2:
                    lon = float(parts[0])
                    lat = float(parts[1])
                    
                    # Determine if start/end
                    is_start = False
                    is_end = False
                    if 'start' in waypoint_name.lower() or 'begin' in waypoint_name.lower():
                        is_start = True
                    if 'end' in waypoint_name.lower() or 'finish' in waypoint_name.lower():
                        is_end = True
                    
                    waypoints.append({
                        'name': waypoint_name,
                        'latitude': lat,
                        'longitude': lon,
                        'description': description,
                        'is_start': is_start,
                        'is_end': is_end
                    })
    
    if excluded_count > 0:
        print(f"  Excluded {excluded_count} elevation waypoints")
    
    print(f"  Found {len(waypoints)} waypoints")
    return waypoints

# ==================== PATH SMOOTHING ====================

def smooth_path(coordinates: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    """
    Smooth jagged corners in the path using spline interpolation.
    Only smooths corners that are sharper than MIN_CORNER_ANGLE_DEG.
    """
    if not ENABLE_SMOOTHING or not SCIPY_AVAILABLE or len(coordinates) < 4:
        return coordinates
    
    print(f"\nSmoothing path (corner threshold: {MIN_CORNER_ANGLE_DEG}°)...")
    
    # Calculate total path length
    total_length = sum(
        haversine_distance(coordinates[i][0], coordinates[i][1],
                         coordinates[i+1][0], coordinates[i+1][1])
        for i in range(len(coordinates) - 1)
    )
    
    # Identify sharp corners
    sharp_corners = []
    for i in range(1, len(coordinates) - 1):
        angle = calculate_angle(coordinates[i-1], coordinates[i], coordinates[i+1])
        if angle < MIN_CORNER_ANGLE_DEG:
            sharp_corners.append(i)
    
    print(f"  Found {len(sharp_corners)} sharp corners to smooth")
    
    if not sharp_corners:
        print("  No sharp corners found - skipping smoothing")
        return coordinates
    
    # Convert to numpy arrays
    lats = np.array([c[0] for c in coordinates])
    lons = np.array([c[1] for c in coordinates])
    
    try:
        # Fit spline
        tck, u = splprep([lats, lons], s=SMOOTHING_FACTOR, k=min(3, len(coordinates)-1))
        
        # Generate new points based on path length
        num_points = int(total_length / 1000 * SMOOTHING_POINTS_PER_KM)
        u_new = np.linspace(0, 1, num_points)
        smooth_lats, smooth_lons = splev(u_new, tck)
        
        # Convert back to list of tuples
        smoothed = list(zip(smooth_lats, smooth_lons))
        
        print(f"  Smoothed path: {len(smoothed)} points (was {len(coordinates)})")
        
        # Calculate path length change
        smoothed_length = sum(
            haversine_distance(smoothed[i][0], smoothed[i][1],
                             smoothed[i+1][0], smoothed[i+1][1])
            for i in range(len(smoothed) - 1)
        )
        
        length_change_pct = ((smoothed_length - total_length) / total_length) * 100
        print(f"  Original length: {total_length/1000:.2f} km")
        print(f"  Smoothed length: {smoothed_length/1000:.2f} km")
        print(f"  Length change: {length_change_pct:+.2f}%")
        
        return smoothed
        
    except Exception as e:
        print(f"  Warning: Smoothing failed ({e})")
        print(f"  Returning original path")
        return coordinates

# ==================== KMZ OUTPUT (matching v7 structure) ====================

def create_kmz_export(coordinates: List[Tuple[float, float]], waypoints: List[Dict], output_path: str):
    """Create KMZ file with path and waypoints (no elevation data)"""
    print(f"\nCreating KMZ export...")
    
    if not SIMPLEKML_AVAILABLE:
        print("  Error: simplekml not installed. Install with: pip install simplekml")
        return
    
    kml = simplekml.Kml(name=f"Stitched Path - {Path(output_path).stem}")
    
    # Main folder
    main_folder = kml.newfolder(name="Route with Stitched Path")
    
    # Create path folder
    path_folder = main_folder.newfolder(name="Stitched Path")
    
    # Single path (no elevation, so z=0)
    coords = [(lon, lat, 0) for lat, lon in coordinates]
    
    path = path_folder.newlinestring(
        name="Stitched Route",
        coords=coords
    )
    path.style.linestyle.color = simplekml.Color.red
    path.style.linestyle.width = 4
    
    # Add waypoints
    if waypoints:
        waypoint_folder = main_folder.newfolder(name="Waypoints")
        
        for wp in waypoints:
            # Determine icon color
            if wp['is_start']:
                icon_color = simplekml.Color.green
                icon_scale = 1.0
            elif wp['is_end']:
                icon_color = simplekml.Color.red
                icon_scale = 1.0
            else:
                icon_color = simplekml.Color.yellow
                icon_scale = 0.7
            
            pnt = waypoint_folder.newpoint(
                name=wp['name'],
                coords=[(wp['longitude'], wp['latitude'], 0)]
            )
            
            if wp.get('description'):
                pnt.description = wp['description']
            
            pnt.style.iconstyle.color = icon_color
            pnt.style.iconstyle.scale = icon_scale
    
    # Save KMZ
    kml.savekmz(output_path)
    print(f"  Saved to: {output_path}")

# ==================== MAIN ====================

def main():
    """Main processing pipeline"""
    print("=" * 70)
    print("KMZ PATH STITCHER")
    print("Modified from Elevation Profile Extraction v7")
    print("=" * 70)
    print(f"Input: {INPUT_KMZ}")
    print(f"Output: {OUTPUT_KMZ}")
    print(f"Smoothing: {'Enabled' if ENABLE_SMOOTHING else 'Disabled'}")
    if ENABLE_SMOOTHING and SCIPY_AVAILABLE:
        print(f"  Corner threshold: {MIN_CORNER_ANGLE_DEG}°")
        print(f"  Points per km: {SMOOTHING_POINTS_PER_KM}")
    print("=" * 70)
    
    try:
        # Step 1: Extract and stitch coordinates (using original v7 function)
        coordinates = extract_coordinates_from_kmz_with_path_fixing(INPUT_KMZ)
        
        if not coordinates:
            print("Error: No coordinates extracted from KMZ file")
            return
        
        # Step 2: Extract waypoints (using original v7 function)
        waypoints = extract_kmz_waypoints(INPUT_KMZ, exclude_elevation_waypoints=True)
        
        # Step 3: Smooth the path (optional)
        if ENABLE_SMOOTHING and SCIPY_AVAILABLE:
            smoothed_coordinates = smooth_path(coordinates)
        else:
            smoothed_coordinates = coordinates
            if ENABLE_SMOOTHING and not SCIPY_AVAILABLE:
                print("\nSmoothing requested but scipy not available")
            else:
                print("\nSmoothing disabled - using raw stitched path")
        
        # Step 4: Create output KMZ with waypoints
        create_kmz_export(smoothed_coordinates, waypoints, str(OUTPUT_KMZ))
        
        print("\n" + "=" * 70)
        print("COMPLETE")
        print("=" * 70)
        print(f"✓ Input coordinates: {len(coordinates)}")
        print(f"✓ Output coordinates: {len(smoothed_coordinates)}")
        print(f"✓ Waypoints preserved: {len(waypoints)}")
        print(f"✓ Output file: {OUTPUT_KMZ.name}")
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()